# Vehicle Re-Identification — Evaluation на VeRi-776

Практическая часть дипломной работы. Сравнение архитектур **ResNet-50** и **ViT-Small** для задачи реидентификации транспортных средств.

**Что делает ноутбук:**
1. Клонирует репозиторий с кодом
2. Скачивает датасет VeRi-776 и предобученные веса
3. Запускает evaluation: базовые метрики Rank-1/5/10, mAP
4. Прогоняет re-ranking ($k_1{=}20,\ k_2{=}6,\ \lambda{=}0.3$)
5. Прогоняет evaluation с расстоянием Махаланобиса
6. Строит сводную таблицу и графики

**Требования:** Runtime → Change runtime type → **T4 GPU** (бесплатный).

**Время прогона:** ~15 минут (включая загрузку данных).

## 1. Подготовка окружения

In [1]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
!pip install -q gdown timm scipy matplotlib pillow numpy tqdm

## 2. Клонирование репозитория

In [3]:
!git clone https://github.com/maksgorbachev/vehicle-reid.git
%cd vehicle-reid/code

Cloning into 'vehicle-reid'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 104 (delta 7), reused 104 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 37.45 MiB | 25.48 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/vehicle-reid/code


## 3. Загрузка датасета VeRi-776 и предобученных весов

Файлы лежат на Google Drive. Используем `gdown` для скачивания по ID.

In [4]:
import os
os.makedirs('checkpoints', exist_ok=True)
os.makedirs('data', exist_ok=True)

# ResNet-50 + BNNeck (95 MB)
!gdown --id 1UZcyEwDSVYnF_TXao6WD9zzWdBVitV0- -O checkpoints/resnet50_best.pth

# ViT-Small + BNNeck (84 MB)
!gdown --id 1fahB0pxmQCmaaR6cJNueACOdDguzcvbm -O checkpoints/vit_best.pth

# VeRi-776 dataset (~290 MB)
!gdown --id 1lkH6YdhsBYT1IiQmWsLKWHWqmPo1HJBZ -O data/VeRi.zip

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1UZcyEwDSVYnF_TXao6WD9zzWdBVitV0-
From (redirected): https://drive.google.com/uc?id=1UZcyEwDSVYnF_TXao6WD9zzWdBVitV0-&confirm=t&uuid=e7110919-9d93-4890-b831-eaa9a852c75d
To: /content/vehicle-reid/code/checkpoints/resnet50_best.pth
100% 99.1M/99.1M [00:00<00:00, 111MB/s]
/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1fahB0pxmQCmaaR6cJNueACOdDguzcvbm
From (redirected): https://drive.google.com/uc?id=1fahB0pxmQCmaaR6cJNueACOdDguzcvbm&confirm=t&uuid=54eaf9b1-46af-4314-a6bf-0ac2

In [5]:
import zipfile
with zipfile.ZipFile('data/VeRi.zip', 'r') as z:
    z.extractall('data/')

# Определяем корень датасета (может быть data/VeRi/ или data/ напрямую)
import os
if os.path.isdir('data/VeRi/image_query'):
    VERI_ROOT = 'data/VeRi'
elif os.path.isdir('data/image_query'):
    VERI_ROOT = 'data'
else:
    raise RuntimeError('VeRi structure not found')

print('VeRi root:', VERI_ROOT)
print('Query images:', len(os.listdir(os.path.join(VERI_ROOT, 'image_query'))))
print('Test images:', len(os.listdir(os.path.join(VERI_ROOT, 'image_test'))))

VeRi root: data/VeRi
Query images: 1678
Test images: 11579


## 4. Evaluation ResNet-50 + BNNeck

Прогоняем модель на 1678 query images против 11579 test images. Считаем CMC и mAP по стандартному протоколу VeRi-776 (исключение same-camera matches).

In [6]:
import sys
sys.path.insert(0, '.')

import torch
from torch.utils.data import DataLoader
from vehicle_reid.model import build_model
from vehicle_reid.vit_model import build_vit_model
from vehicle_reid.dataset import VeRiDataset
from vehicle_reid.transforms import get_test_transforms
from vehicle_reid.evaluation import ReIDEvaluator

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def load_resnet50(ckpt_path):
    model = build_model(num_classes=576, pretrained=False, last_stride=1)
    state = torch.load(ckpt_path, map_location=device)
    if 'model_state_dict' in state:
        state = state['model_state_dict']
    model.load_state_dict(state, strict=False)
    return model.to(device).eval()

def load_vit(ckpt_path):
    model = build_vit_model(num_classes=576, vit_size='small', pretrained=False)
    state = torch.load(ckpt_path, map_location=device)
    if 'model_state_dict' in state:
        state = state['model_state_dict']
    model.load_state_dict(state, strict=False)
    return model.to(device).eval()

def make_loaders(veri_root, batch_size=64):
    tfm = get_test_transforms()
    query_ds = VeRiDataset(root=veri_root, split='query', transform=tfm)
    gallery_ds = VeRiDataset(root=veri_root, split='gallery', transform=tfm)
    query_loader = DataLoader(query_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    gallery_loader = DataLoader(gallery_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    return query_loader, gallery_loader

In [7]:
print('Loading ResNet-50...')
resnet = load_resnet50('checkpoints/resnet50_best.pth')
query_loader, gallery_loader = make_loaders(VERI_ROOT)

evaluator = ReIDEvaluator(resnet, device=device, use_flip=True)
resnet_results = evaluator.evaluate(query_loader, gallery_loader)
evaluator.print_results(resnet_results, title='ResNet-50 + BNNeck')
del resnet; torch.cuda.empty_cache()

Loading ResNet-50...


RuntimeError: Error(s) in loading state_dict for VehicleReIDModel:
	size mismatch for bnneck.classifier.weight: copying a param with shape torch.Size([575, 2048]) from checkpoint, the shape in current model is torch.Size([576, 2048]).

## 5. Evaluation ViT-Small + BNNeck

In [ ]:
print('Loading ViT-Small...')
vit = load_vit('checkpoints/vit_best.pth')

evaluator = ReIDEvaluator(vit, device=device, use_flip=True)
vit_results = evaluator.evaluate(query_loader, gallery_loader)
evaluator.print_results(vit_results, title='ViT-Small + BNNeck')
del vit; torch.cuda.empty_cache()

## 6. Re-ranking (k-reciprocal encoding)

Постобработка результатов retrieval методом k-reciprocal nearest neighbors (Zhong et al., CVPR 2017). Параметры: $k_1=20$, $k_2=6$, $\lambda=0.3$.

In [ ]:
!python scripts/eval_reranking.py \
    --model_type resnet50 \
    --checkpoint checkpoints/resnet50_best.pth \
    --data_dir $VERI_ROOT \
    --output results/rr_resnet50_live.json

In [ ]:
!python scripts/eval_reranking.py \
    --model_type vit \
    --checkpoint checkpoints/vit_best.pth \
    --data_dir $VERI_ROOT \
    --output results/rr_vit_live.json

## 7. Mahalanobis distance

Замена L2 расстояния на Махаланобисово (учитывает ковариацию признаков).

In [ ]:
!python scripts/eval_mahalanobis.py \
    --resnet_ckpt checkpoints/resnet50_best.pth \
    --vit_ckpt checkpoints/vit_best.pth \
    --data_dir $VERI_ROOT \
    --out results/mahalanobis_live.json

## 8. Сводная таблица результатов

In [ ]:
import json

# Прогруженные ранее результаты + результаты из репо (для метрик которые здесь не перепрогоняли)
with open('results/results_resnet50.json') as f:
    saved_resnet = json.load(f)
with open('results/results_vit.json') as f:
    saved_vit = json.load(f)
with open('results/results_resnet50_rr.json') as f:
    rr_resnet = json.load(f)
with open('results/results_vit_rr.json') as f:
    rr_vit = json.load(f)
with open('results/mahalanobis_results.json') as f:
    maha = json.load(f)
with open('results/results_yolo_crop.json') as f:
    yolo = json.load(f)

def fmt(v):
    return f'{v:.2f}' if isinstance(v, (int, float)) else str(v)

rows = [
    ('ResNet-50 + BNNeck (L2)',
        resnet_results['Rank-1'], resnet_results['Rank-5'], resnet_results['mAP']),
    ('ResNet-50 + Re-ranking',
        rr_resnet.get('Rank-1', 0), rr_resnet.get('Rank-5', 0), rr_resnet.get('mAP', 0)),
    ('ResNet-50 + Mahalanobis',
        maha['resnet50']['mahalanobis']['Rank-1'],
        maha['resnet50']['mahalanobis']['Rank-5'],
        maha['resnet50']['mahalanobis']['mAP']),
    ('ViT-Small + BNNeck (L2)',
        vit_results['Rank-1'], vit_results['Rank-5'], vit_results['mAP']),
    ('ViT-Small + Re-ranking',
        rr_vit.get('Rank-1', 0), rr_vit.get('Rank-5', 0), rr_vit.get('mAP', 0)),
    ('ViT-Small + YOLOv11 crop',
        yolo['yolo_crop']['Rank-1'], yolo['yolo_crop']['Rank-5'], yolo['yolo_crop']['mAP']),
]

print('=' * 75)
print(f'{"Метод":<32} {"Rank-1":>10} {"Rank-5":>10} {"mAP":>10}')
print('-' * 75)
for name, r1, r5, mp in rows:
    print(f'{name:<32} {fmt(r1):>10} {fmt(r5):>10} {fmt(mp):>10}')
print('=' * 75)

## 9. График сравнения

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

names = [r[0] for r in rows]
rank1 = [r[1] for r in rows]
rank5 = [r[2] for r in rows]
maps = [r[3] for r in rows]

x = np.arange(len(names))
width = 0.27

fig, ax = plt.subplots(figsize=(13, 6))
ax.bar(x - width, rank1, width, label='Rank-1', color='#2ecc71')
ax.bar(x,         rank5, width, label='Rank-5', color='#3498db')
ax.bar(x + width, maps,  width, label='mAP',    color='#e74c3c')

ax.set_ylabel('Метрика (%)')
ax.set_title('Vehicle ReID на VeRi-776: сравнение методов')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=20, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, 100)

plt.tight_layout()
plt.savefig('comparison.png', dpi=150)
plt.show()

## 10. Выводы

- **ViT-Small превосходит ResNet-50** по всем основным метрикам — Rank-1 90.0% vs 88.7%, mAP 69.4% vs 64.3% — при меньшем числе параметров (21.9M против 24.7M).
- **Re-ranking** даёт стабильный прирост mAP: +3.2% у ResNet-50, +5.1% у ViT-Small. ViT выигрывает сильнее благодаря более компактным кластерам в пространстве эмбеддингов.
- **Mahalanobis distance** улучшает результаты ResNet-50 (+0.28% mAP), но ухудшает ViT-Small ($-31\%$ mAP). Это объясняется размерностью эмбеддинга: для ViT (D=384) полная ковариация неустойчива на ограниченном train-наборе.
- **YOLOv11 detection → crop → ReID** показывает реалистичный сценарий: при 21.5% fallback (детектор не нашёл TS) mAP падает с 69.4% до 65.5%. Это деградация в пределах нормы для end-to-end пайплайна.

Полное описание методологии и результатов — в тексте диплома, главы 2 и 3.